# 07 Plot Results

Goal:

1. read cleaned csv outputs from `results/part1/`
2. produce publication-friendly figures for Part 1
3. separate main figures from appendix figures
4. keep plotting logic readable and easy to tweak

Recommended narrative:
- main: final test metrics + test PSNR convergence
- appendix: test L1 convergence + train/test overlay diagnostics


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CV_ROOT = Path('~/CV_Project').expanduser()
RESULTS_ROOT = CV_ROOT / 'results' / 'part1'
PLOTS_ROOT = CV_ROOT / 'plots' / 'part1'
MAIN_DIR = PLOTS_ROOT / 'main'
APP_DIR = PLOTS_ROOT / 'appendix'
TABLE_DIR = PLOTS_ROOT / 'tables'

for d in [MAIN_DIR, APP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FINAL_CSV = RESULTS_ROOT / 'final' / 'final_metrics_18.csv'
CONV_CSV = RESULTS_ROOT / 'convergence' / 'all_convergence_metrics.csv'

final_df = pd.read_csv(FINAL_CSV)
conv_df = pd.read_csv(CONV_CSV)

print('final_df shape =', final_df.shape)
print('conv_df shape =', conv_df.shape)


final_df shape = (18, 11)
conv_df shape = (720, 11)


## 1. Styling and labels


In [4]:
SCENE_ORDER = ['Re10k-1', 'DL3DV-2', '405841']
SETTING_ORDER = ['planA_colmap_full', 'planA_colmap_96', 'planB_vggt_96']
MODEL_ORDER = ['3dgs', 'scaffold']
METRIC_ORDER = ['ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']

SETTING_LABELS = {
    'planA_colmap_full': 'Plan A - COLMAP full',
    'planA_colmap_96': 'Plan A - COLMAP-96',
    'planB_vggt_96': 'Plan B - VGGT-96',
}
MODEL_LABELS = {
    '3dgs': '3DGS',
    'scaffold': 'Scaffold-GS',
}
SCENE_LABELS = {
    'Re10k-1': 'Re10k-1',
    'DL3DV-2': 'DL3DV-2',
    '405841': '405841',
}
SETTING_COLORS = {
    'planA_colmap_full': '#1f4e79',
    'planA_colmap_96': '#2e8b57',
    'planB_vggt_96': '#8b1e3f',
}
MODEL_LINESTYLES = {
    '3dgs': '-',
    'scaffold': '--',
}
MODEL_MARKERS = {
    '3dgs': 'o',
    'scaffold': 'D',
}

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def add_setting_key(df):
    df = df.copy()
    df['setting'] = df['plan'] + '_' + df['variant']
    return df

final_df = add_setting_key(final_df)
conv_df = add_setting_key(conv_df)


## 2. Quick sanity checks


In [5]:
display(final_df[['scene', 'setting', 'model', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']].sort_values(['scene', 'setting', 'model']))
print(conv_df.groupby(['scene', 'setting', 'model', 'split'])['iter'].agg(['min', 'max', 'count']).reset_index().to_string(index=False))


,scene,setting,model,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,405841,planA_colmap_96,3dgs,31.417475,0.930650,0.124798
1,405841,planA_colmap_96,scaffold,31.730192,0.931433,0.124264
2,405841,planA_colmap_full,3dgs,32.770302,0.937703,0.125829
3,405841,planA_colmap_full,scaffold,32.061390,0.933910,0.135362
4,405841,planB_vggt_96,3dgs,30.603487,0.925874,0.128766
5,405841,planB_vggt_96,scaffold,31.054298,0.925593,0.128784
6,DL3DV-2,planA_colmap_96,3dgs,31.171646,0.941346,0.095407
7,DL3DV-2,planA_colmap_96,scaffold,34.504063,0.963633,0.066687
8,DL3DV-2,planA_colmap_full,3dgs,34.760578,0.965748,0.067786
9,DL3DV-2,planA_colmap_full,scaffold,36.181946,0.971939,0.056363


  scene           setting    model split  min   max  count
 405841   planA_colmap_96     3dgs  test 2000 40000     20
 405841   planA_colmap_96     3dgs train 2000 40000     20
 405841   planA_colmap_96 scaffold  test 2000 40000     20
 405841   planA_colmap_96 scaffold train 2000 40000     20
 405841 planA_colmap_full     3dgs  test 2000 40000     20
 405841 planA_colmap_full     3dgs train 2000 40000     20
 405841 planA_colmap_full scaffold  test 2000 40000     20
 405841 planA_colmap_full scaffold train 2000 40000     20
 405841     planB_vggt_96     3dgs  test 2000 40000     20
 405841     planB_vggt_96     3dgs train 2000 40000     20
 405841     planB_vggt_96 scaffold  test 2000 40000     20
 405841     planB_vggt_96 scaffold train 2000 40000     20
DL3DV-2   planA_colmap_96     3dgs  test 2000 40000     20
DL3DV-2   planA_colmap_96     3dgs train 2000 40000     20
DL3DV-2   planA_colmap_96 scaffold  test 2000 40000     20
DL3DV-2   planA_colmap_96 scaffold train 2000 40000     

## 3. Export a tidy summary table


In [6]:
summary_cols = ['scene', 'plan', 'variant', 'setting', 'model', 'run_name', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']
summary_df = final_df[summary_cols].sort_values(['scene', 'setting', 'model']).reset_index(drop=True)
summary_out = TABLE_DIR / 'part1_final_metrics_summary.csv'
summary_df.to_csv(summary_out, index=False)
print('saved', summary_out)
summary_df


saved /home/bzhang512/CV_Project/plots/part1/tables/part1_final_metrics_summary.csv


,scene,plan,variant,setting,model,run_name,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,405841,planA,colmap_96,planA_colmap_96,3dgs,default_40k_eval2k,31.417475,0.930650,0.124798
1,405841,planA,colmap_96,planA_colmap_96,scaffold,final_40k_eval2k,31.730192,0.931433,0.124264
2,405841,planA,colmap_full,planA_colmap_full,3dgs,default_40k_eval2k,32.770302,0.937703,0.125829
3,405841,planA,colmap_full,planA_colmap_full,scaffold,final_40k_eval2k,32.061390,0.933910,0.135362
4,405841,planB,vggt_96,planB_vggt_96,3dgs,default_40k_eval2k,30.603487,0.925874,0.128766
5,405841,planB,vggt_96,planB_vggt_96,scaffold,final_40k_eval2k,31.054298,0.925593,0.128784
6,DL3DV-2,planA,colmap_96,planA_colmap_96,3dgs,default_40k_eval2k,31.171646,0.941346,0.095407
7,DL3DV-2,planA,colmap_96,planA_colmap_96,scaffold,final_40k_eval2k,34.504063,0.963633,0.066687
8,DL3DV-2,planA,colmap_full,planA_colmap_full,3dgs,default_40k_eval2k,34.760578,0.965748,0.067786
9,DL3DV-2,planA,colmap_full,planA_colmap_full,scaffold,final_40k_eval2k,36.181946,0.971939,0.056363


## 4. Main figure: final test PSNR / SSIM / LPIPS


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
metrics = [
    ('ours_40000.PSNR', 'PSNR ↑'),
    ('ours_40000.SSIM', 'SSIM ↑'),
    ('ours_40000.LPIPS', 'LPIPS ↓'),
]
bar_width = 0.22
x = np.arange(len(SCENE_ORDER))

for ax, (metric, ylabel) in zip(axes, metrics):
    for i, setting in enumerate(SETTING_ORDER):
        offset_base = (i - 1) * 2 * bar_width
        for j, model in enumerate(MODEL_ORDER):
            xpos = x + offset_base + j * bar_width
            vals = []
            for scene in SCENE_ORDER:
                row = final_df[(final_df['scene'] == scene) & (final_df['setting'] == setting) & (final_df['model'] == model)]
                vals.append(row.iloc[0][metric] if not row.empty else np.nan)
            ax.bar(
                xpos,
                vals,
                width=bar_width,
                color=SETTING_COLORS[setting],
                alpha=0.95 if model == '3dgs' else 0.55,
                edgecolor='black' if model == 'scaffold' else 'none',
                linewidth=0.8,
            )
    ax.set_xticks(x)
    ax.set_xticklabels([SCENE_LABELS[s] for s in SCENE_ORDER])
    ax.set_ylabel(ylabel)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.3)
    ax.set_axisbelow(True)
    ax.set_title(ylabel)

setting_handles = [
    Line2D([0], [0], color=SETTING_COLORS[s], lw=6, label=SETTING_LABELS[s].replace('', '\n'))
    for s in SETTING_ORDER
]
model_handles = [
    Line2D([0], [0], color='gray', lw=8, alpha=0.95 if m == '3dgs' else 0.55,
           marker='s', markersize=8, label=MODEL_LABELS[m])
    for m in MODEL_ORDER
]

leg1 = axes[0].legend(handles=setting_handles, frameon=False, title='Input setting', loc='best')
axes[0].add_artist(leg1)
axes[2].legend(handles=model_handles, frameon=False, title='Method', loc='best')

fig.suptitle('Part 1 Final Test Metrics', y=1.03, fontsize=14)
fig.tight_layout()
out = MAIN_DIR / 'part1_final_test_metrics.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
plt.close(fig)
print('saved', out)


saved /home/bzhang512/CV_Project/plots/part1/main/part1_final_test_metrics.png


## 5. Main figure: test PSNR convergence (3 x 3 grid)


In [10]:
fig, axes = plt.subplots(len(SCENE_ORDER), len(SETTING_ORDER), figsize=(14, 10), sharex=True)
for r, scene in enumerate(SCENE_ORDER):
    for c, setting in enumerate(SETTING_ORDER):
        ax = axes[r, c]
        sub = conv_df[(conv_df['scene'] == scene) & (conv_df['setting'] == setting) & (conv_df['split'] == 'test')].copy()
        for model in MODEL_ORDER:
            cur = sub[sub['model'] == model].sort_values('iter')
            if cur.empty:
                continue
            ax.plot(
                cur['iter'], cur['psnr'],
                color=SETTING_COLORS[setting],
                linestyle=MODEL_LINESTYLES[model],
                marker=MODEL_MARKERS[model],
                linewidth=2.0,
                markersize=4,
                label=MODEL_LABELS[model],
            )
        if r == 0:
            ax.set_title(SETTING_LABELS[setting])
        if c == 0:
            ax.set_ylabel(f'{SCENE_LABELS[scene]}\n PSNR ↑')
        ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
        ax.set_axisbelow(True)

for ax in axes[-1, :]:
    ax.set_xlabel('Iteration')

method_handles = [
    Line2D([0], [0], color='black', linestyle=MODEL_LINESTYLES[m], marker=MODEL_MARKERS[m], lw=2.0, label=MODEL_LABELS[m])
    for m in MODEL_ORDER
]
fig.legend(handles=method_handles, frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(0.5, 1.01))
fig.suptitle('Part 1 Test PSNR Convergence', y=1.04, fontsize=14)
fig.tight_layout()
out = MAIN_DIR / 'part1_test_psnr_convergence_grid.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
plt.close(fig)
print('saved', out)


saved /home/bzhang512/CV_Project/plots/part1/main/part1_test_psnr_convergence_grid.png


## 6. Appendix figure: test L1 convergence (3 x 3 grid)


In [11]:
fig, axes = plt.subplots(len(SCENE_ORDER), len(SETTING_ORDER), figsize=(14, 10), sharex=True)
for r, scene in enumerate(SCENE_ORDER):
    for c, setting in enumerate(SETTING_ORDER):
        ax = axes[r, c]
        sub = conv_df[(conv_df['scene'] == scene) & (conv_df['setting'] == setting) & (conv_df['split'] == 'test')].copy()
        for model in MODEL_ORDER:
            cur = sub[sub['model'] == model].sort_values('iter')
            if cur.empty:
                continue
            ax.plot(
                cur['iter'], cur['l1'],
                color=SETTING_COLORS[setting],
                linestyle=MODEL_LINESTYLES[model],
                marker=MODEL_MARKERS[model],
                linewidth=2.0,
                markersize=4,
                label=MODEL_LABELS[model],
            )
        if r == 0:
            ax.set_title(SETTING_LABELS[setting])
        if c == 0:
            ax.set_ylabel(f'{SCENE_LABELS[scene]}\n L1 ↓')
        ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
        ax.set_axisbelow(True)

for ax in axes[-1, :]:
    ax.set_xlabel('Iteration')

method_handles = [
    Line2D([0], [0], color='black', linestyle=MODEL_LINESTYLES[m], marker=MODEL_MARKERS[m], lw=2.0, label=MODEL_LABELS[m])
    for m in MODEL_ORDER
]
fig.legend(handles=method_handles, frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(0.5, 1.01))
fig.suptitle('Part 1 Test L1 Convergence', y=1.04, fontsize=14)
fig.tight_layout()
out = APP_DIR / 'part1_test_l1_convergence_grid.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
plt.close(fig)
print('saved', out)


saved /home/bzhang512/CV_Project/plots/part1/appendix/part1_test_l1_convergence_grid.png


## 7. Appendix figure: train/test overlay diagnostics


In [12]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
scene = 'Re10k-1'
for c, setting in enumerate(SETTING_ORDER):
    ax_psnr = axes[0, c]
    ax_l1 = axes[1, c]
    sub = conv_df[(conv_df['scene'] == scene) & (conv_df['setting'] == setting)].copy()
    for model in MODEL_ORDER:
        for split, alpha in [('test', 1.0), ('train', 0.55)]:
            cur = sub[(sub['model'] == model) & (sub['split'] == split)].sort_values('iter')
            if cur.empty:
                continue
            ax_psnr.plot(
                cur['iter'], cur['psnr'],
                color=SETTING_COLORS[setting], linestyle=MODEL_LINESTYLES[model],
                linewidth=2.0 if split == 'test' else 1.6, alpha=alpha,
                marker=MODEL_MARKERS[model] if split == 'test' else None, markersize=3.5,
            )
            ax_l1.plot(
                cur['iter'], cur['l1'],
                color=SETTING_COLORS[setting], linestyle=MODEL_LINESTYLES[model],
                linewidth=2.0 if split == 'test' else 1.6, alpha=alpha,
                marker=MODEL_MARKERS[model] if split == 'test' else None, markersize=3.5,
            )
    ax_psnr.set_title(SETTING_LABELS[setting])
    ax_psnr.set_ylabel('PSNR ↑')
    ax_l1.set_ylabel('L1 ↓')
    ax_l1.set_xlabel('Iteration')
    ax_psnr.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
    ax_l1.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
    ax_psnr.set_axisbelow(True)
    ax_l1.set_axisbelow(True)

method_handles = [
    Line2D([0], [0], color='black', linestyle=MODEL_LINESTYLES[m], marker=MODEL_MARKERS[m], lw=2.0, label=MODEL_LABELS[m])
    for m in MODEL_ORDER
]
split_handles = [
    Line2D([0], [0], color='black', lw=2.0, alpha=1.0, label='Test'),
    Line2D([0], [0], color='black', lw=2.0, alpha=0.55, label='Train'),
]
leg1 = axes[0, 0].legend(handles=method_handles, frameon=False, title='Method', loc='best')
axes[0, 0].add_artist(leg1)
axes[1, 2].legend(handles=split_handles, frameon=False, title='Split', loc='best')
fig.suptitle(f'Appendix: Train/Test Overlay Diagnostics ({scene})', y=1.02, fontsize=14)
fig.tight_layout()
out = APP_DIR / 'part1_train_test_overlay_re10k1.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
plt.close(fig)
print('saved', out)


saved /home/bzhang512/CV_Project/plots/part1/appendix/part1_train_test_overlay_re10k1.png


## 8. Optional aggregate ranking table


In [13]:
agg_df = final_df.groupby(['setting', 'model'], as_index=False).agg({
    'ours_40000.PSNR': 'mean',
    'ours_40000.SSIM': 'mean',
    'ours_40000.LPIPS': 'mean',
})
agg_df = agg_df.sort_values(['setting', 'model']).reset_index(drop=True)
agg_out = TABLE_DIR / 'part1_aggregate_mean_metrics.csv'
agg_df.to_csv(agg_out, index=False)
print('saved', agg_out)
agg_df


saved /home/bzhang512/CV_Project/plots/part1/tables/part1_aggregate_mean_metrics.csv


,setting,model,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,planA_colmap_96,3dgs,32.012523,0.946796,0.090882
1,planA_colmap_96,scaffold,33.378983,0.955641,0.079810
2,planA_colmap_full,3dgs,34.231937,0.960346,0.078676
3,planA_colmap_full,scaffold,34.368463,0.960927,0.078524
4,planB_vggt_96,3dgs,29.922806,0.917640,0.114839
5,planB_vggt_96,scaffold,29.970181,0.918573,0.116348


## 9. Show generated files


In [14]:
for p in sorted(MAIN_DIR.glob('*')):
    print('MAIN    ', p.name)
for p in sorted(APP_DIR.glob('*')):
    print('APPENDIX', p.name)
for p in sorted(TABLE_DIR.glob('*')):
    print('TABLE   ', p.name)


MAIN     part1_final_test_metrics.png
MAIN     part1_test_psnr_convergence_grid.png
APPENDIX part1_test_l1_convergence_grid.png
APPENDIX part1_train_test_overlay_re10k1.png
TABLE    part1_aggregate_mean_metrics.csv
TABLE    part1_final_metrics_summary.csv
